# 1. PTQ và QAT

PTQ INT8: Lấy mô hình FP32 đã huấn luyện, chọn 50 mẫu đại diện cho mỗi lớp từ tập Train để hiệu chuẩn lượng tử, sau đó chuyển sang INT8. Không huấn luyện lại trọng số.

QAT INT8: Lấy cùng checkpoint FP32, sao chép trọng số, chèn cơ chế fake quantization vào các lớp phù hợp rồi tiếp tục fine-tuning. Cụ thể, script đang dùng:
- Learning rate 1e-5, tối đa 15 epoch, batch size 32.
- Adam và Binary Crossentropy.
- Early Stopping dựa trên val_loss, khôi phục trọng số tốt nhất.
- Class weighting được tính từ tập Train.
- Sau fine-tuning, vẫn dùng tập Train đại diện để chuyển mô hình sang INT8.

### Lý thuyết trung về 2 phương pháp lượng tử

| Tiêu chí | PTQ | QAT |
|---|---|---|
| Tên đầy đủ | Post-Training Quantization | Quantization-Aware Training |
| Thời điểm | Sau huấn luyện | Trong huấn luyện/tinh chỉnh |
| Có cập nhật trọng số ? | Không | Có |
| Cách hoạt động | Chuyển FP32 sang INT8, hiệu chuẩn dải giá trị | Mô phỏng sai số INT8 khi huấn luyện |
| Ưu | Nhanh, đơn giản | Có khả năng thích nghi với sai số lượng tử |
| Nhược | Có thể giảm Accuracy/Recall | Tốn công huấn luyện |

PTQ giữ nguyên trọng số đã học về mặt giá trị gốc, rồi ánh xạ chúng về biểu diễn số nguyên có độ chính xác thấp hơn
> Do sai số làm tròn, đầu ra của mạng có thể thay đổi. PTQ không huấn luyện lại để sửa những thay đổi đó.

QAT đưa sai số lượng tử hóa vào quá trình huấn luyện. Mô hình học cách điều chỉnh trọng số để giảm tác động của sai số đó
> **Điểm quan trọng:** Trong QAT thông thường, trọng số vẫn được tối ưu bằng số thực trong khi mô phỏng phép lượng tử hóa, không phải toàn bộ quá trình huấn luyện đều thực hiện bằng INT8.

# 2. Câu hỏi:

**Tại sao mô hình QAT INT8 có thể đạt Accuracy cao hơn FP32 trong khi độ chính xác biểu diễn số giảm từ 32 bit xuống 8 bit?**

Việc giảm độ chính xác biểu diễn số không đồng nghĩa với việc Accuracy phân loại bắt buộc giảm.

Trong nghiên cứu này, PTQ thực hiện lượng tử hóa sau huấn luyện mà không cập nhật trọng số, trong khi QAT tiếp tục fine-tuning từ checkpoint FP32 với cơ chế mô phỏng sai số lượng tử hóa. Quá trình này cho phép mô hình điều chỉnh trọng số và ranh giới quyết định để thích nghi với biểu diễn INT8.

Trên 24 checkpoint khảo sát, QAT INT8 đạt Accuracy trung bình trên historical Test là 91,54%, cao hơn FP32 là 90,54%, trong khi Accuracy trung bình trên Validation gần tương đương, lần lượt là 97,24% và 97,35%. Cả ba phương pháp đều được đánh giá với ngưỡng phân loại cố định 0,5.

Kết quả này cho thấy QAT có khả năng duy trì hiệu năng và trong một số trường hợp còn cải thiện kết quả phân loại sau lượng tử hóa. Tuy nhiên, nhóm không khẳng định INT8 ưu việt hơn FP32 về khả năng biểu diễn hay toàn bộ cải thiện đến từ riêng kỹ thuật lượng tử hóa, bởi quá trình fine-tuning cũng có thể đóng góp vào sự thay đổi hiệu năng.

Giá trị chính của kết quả là mô hình INT8 đạt chất lượng phân loại cạnh tranh với FP32 trong khi giảm đáng kể nhu cầu lưu trữ, tạo điều kiện triển khai thực tế trên ESP32-S3.

# 3. Mục tiêu

Mục tiêu của nhóm khi áp dụng lượng tử hóa không phải làm cho mô hình INT8 chính xác hơn FP32, mà là giảm yêu cầu tài nguyên để triển khai trên ESP32-S3 trong khi vẫn duy trì hiệu năng phân loại.

Kết quả khảo sát cho thấy PTQ có thể gây suy giảm đáng kể khả năng nhận diện nhóm Asthma, thể hiện qua Asthma Recall trung bình giảm từ 89,47% xuống 80,59%.

Trong khi đó, QAT sử dụng quá trình fine-tuning có mô phỏng sai số lượng tử hóa, giúp mô hình thích nghi với biểu diễn INT8. Nhờ vậy, Asthma Recall được duy trì ở 89,04%, gần tương đương FP32 và cao hơn PTQ 8,45 điểm phần trăm.

Đồng thời, QAT INT8 đạt Accuracy trung bình 91,54%, cao nhất trong ba phương pháp được khảo sát.

Do đó, nhóm lựa chọn QAT INT8 vì phương pháp này tạo ra sự cân bằng tốt giữa tài nguyên triển khai và hiệu năng phân loại, đặc biệt là khả năng hạn chế bỏ sót các mẫu Asthma.